In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.signal import find_peaks

In [ ]:
df = pd.read_csv("SN_m_tot_V2.0.csv",delimiter=';',decimal='.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# 1. Smooth the data
x = df["rok_norm"].values
y = df["sunspot"].values

smoothed = lowess(y, x, frac=0.01)
df["sunspot_lowess"] = smoothed[:, 1]

# 2. Find peaks directly (much more robust than sign-of-derivative counting)
# distance= enforces a minimum spacing between peaks (in samples) to avoid
# picking up small wiggles as separate cycles. Solar cycles are ~11 years,
# so if you have monthly data, that's ~132 months minimum between real peaks.
peak_idx, properties = find_peaks(df["sunspot_lowess"], distance=60, prominence=5)

peak_times = df["rok_norm"].values[peak_idx]
peak_values = df["sunspot_lowess"].values[peak_idx]

# 3. Periods between consecutive peaks
periods = np.diff(peak_times)

print("Peak years:", peak_times)
print("Periods between peaks (years):", periods)
print("\nStatistics:")
print(f"Mean period: {periods.mean():.2f} years")
print(f"Std dev: {periods.std():.2f} years")
print(f"Min: {periods.min():.2f}, Max: {periods.max():.2f}")

# 4. Plot to sanity-check
plt.figure(figsize=(16,4))
plt.plot(x, y, alpha=0.3, label="Raw sunspot number", color="gray")
plt.plot(x, df["sunspot_lowess"], color="red", label="LOWESS smoothed")
plt.plot(peak_times, peak_values, "o", color="black", label="Detected peaks", markersize=6)

plt.xlabel("Year")
plt.ylabel("International Sunspot Number")
plt.legend()
plt.grid(ls=":")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(periods, bins=15, color="steelblue", edgecolor="black", alpha=0.8)
plt.xlabel("Period between peaks (years)")
plt.ylabel("Count")
plt.title("Distribution of solar cycle periods")
plt.grid(ls=":", alpha=0.5)
plt.axvline(periods.mean(), color="red", linestyle="--", label=f"Mean: {periods.mean():.1f} yr")
plt.axvline(11, color="black", linestyle=":", label="Known solar cycle (~11 yr)")
plt.legend()
plt.tight_layout()
plt.show()